Data Science Using Python and R: Chapter 14 - Page 211: Questions #11, 12, 13, 14, & 15

Use ONLY R for these questions.

For the following exercises, work with the Churn_Training_File data set. Use R to solve each problem. 

In [1]:
from IPython.core.magic import register_cell_magic
from IPython.display import Image, display
import subprocess, tempfile, os

R_STATE_FILE = tempfile.gettempdir().replace('\\', '/') + '/r_state.RData'

@register_cell_magic
def R(line, cell):
    tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    plot_file = tmp.name.replace('\\', '/')
    tmp.close()

    r_code = f"""
    if (file.exists('{R_STATE_FILE}')) load('{R_STATE_FILE}')
    png('{plot_file}')
    {cell}
    invisible(dev.off())
    save.image(file='{R_STATE_FILE}')
    """

    result = subprocess.run(['Rscript', '-e', r_code], capture_output=True, text=True)

    if result.stdout:
        lines = [line for line in result.stdout.split('\n')
                if line.strip() and 'null device' not in line.lower()
                and line.strip() not in ['1', '[1] 1']]
        output = '\n'.join(lines)
        if output.strip():
            print(output)

    if result.stderr and 'Warning' not in result.stderr:
        print("Errors:", result.stderr)

    # Only display if PNG actually has content (threshold raised to catch real plots)
    if os.path.exists(plot_file) and os.path.getsize(plot_file) > 1000:
        display(Image(filename=plot_file))
    os.unlink(plot_file)

print("%%R ready")

%%R ready


Question 11. Subset the variables VMail Plan, Int'll Plan, CustServ Calls, and Churn into their own data frame. Change
CustServ Calls into an ordered factor.

In [2]:
%%R
churn <- read.csv('C:/Users/samsc/Desktop/ADS-502-Hands-On-Exercises/Data_Sets/Churn_Training_File.csv')

In [3]:
%%R
min.churn <- subset(churn, select = c("Intl.Plan", "VMail.Plan", "CustServ.Calls", "Churn"))

In [4]:
%%R
min.churn$CustServ.Calls <- ordered(as.factor(min.churn$CustServ.Calls))

12. Create tables for each of the four variables. Include both counts and proportions in each table. Use the tables to discuss the "baseline" distribution of each variable. 

In [5]:
%%R
VMail_Table <- table(min.churn$VMail.Plan)
VMail_Table <- rbind(VMail_Table, round(prop.table(VMail_Table), 4))
VMail_Table

In [6]:
%%R
colnames(VMail_Table) <- c("VMail.Plan = no", "VMail.Plan = yes")
rownames(VMail_Table) <- c("Count", "Proportion")
VMail_Table

In [7]:
%%R
IntlPlan_Table <- table(min.churn$Intl.Plan)
IntlPlan_Table <- rbind(IntlPlan_Table, round(prop.table(IntlPlan_Table), 4))
IntlPlan_Table

In [8]:
%%R
colnames(IntlPlan_Table) <- c("Intl.Plan = no", "Intl.Plan = yes")
rownames(IntlPlan_Table) <- c("Count", "Proportion")
IntlPlan_Table

In [9]:
%%R
Churn_Table <- table(min.churn$Churn)
Churn_Table <- rbind(Churn_Table, round(prop.table(Churn_Table), 4))
Churn_Table

In [10]:
%%R
colnames(Churn_Table) <- c("Churn = False", "Churn = True")
rownames(Churn_Table) <- c("Count", "Proportion")
Churn_Table

In [11]:
%%R
CustServCalls_Table <- table(min.churn$CustServ.Calls)
CustServCalls_Table <- rbind(CustServCalls_Table, round(prop.table(CustServCalls_Table), 4))
CustServCalls_Table

13. Obtain the association rules using the settings outlined in Section 14.4

In [12]:
import subprocess
result = subprocess.run(['Rscript', '-e', "install.packages('arules', repos='https://cran.r-project.org', type='binary')"],
                        capture_output=True, text=True)

In [14]:
%%R
library(arules)
all.rules <- apiori(data = min.churn, parameter = list(supp = 0.1, target = "rules", conf = 0.4, minlen = 2, maxlen = 2))
inspect(head(all.rules, by = "lift", n= 10))

14. Subset the rules from the previous exercise so none of the antecedents contain Churn variable. Display the rules, sorted by descending lift value. 

In [15]:
%%R
all.rules.ant.df <- as(as(attr(all.rules, "lhs"), "transactions"), "data.frame")

In [ ]:
%%R
rules.dataframe <- as(all.rules.ant.df, "data.frame")

In [ ]:
%%R
t1 <- rules.dataframe$items == "{Churn = True}"
t2 <- rules.dataframe$items == "{Churn = False}"
t1 <- as.integer(t1)
t2 <- as.integer(t2)
non.churn.ant <- abs(t1 + t2 - 1)

In [ ]:
$$R
valid.rules <- all.rules[non.churn.ant == 1]
inspect(head(valid.rules, by = "lift", n = 28))